In [ ]:
# 'Hey Cortana' wake-word training — Kaggle edition.
#
# BEFORE RUNNING (notebook Settings, right sidebar):
#   * Accelerator: GPU T4 x2 (or P100)
#   * Internet:    ON  (requires a phone-verified Kaggle account)
#
# The whole job lives under /kaggle/tmp — the ~60 GiB ephemeral disk — because
# /kaggle/working is capped at ~20 GB and fills up (that cap is why the first
# attempt died at 86% of a 17.3 GB download). Only the 3 final .onnx files
# (a few MB) are copied to /kaggle/working at the end.
#
# Tip for phones: run cells 1-4 interactively to verify, then use
# "Save Version -> Save & Run All (Commit)" — it executes headless on Kaggle's
# side for up to ~12 h even with your browser closed.
!mkdir -p /kaggle/tmp
!df -h /kaggle/working /kaggle/tmp
print('--- expect the /kaggle/tmp mount to show ~55G+ available ---')

In [ ]:
# Point EVERY cache and download at the big disk BEFORE anything installs.
# os.environ persists for this kernel and is inherited by later ! commands.
import os
BIG = '/kaggle/tmp'
os.environ['HF_HOME'] = f'{BIG}/hf'                    # HuggingFace caches (the 17.3 GB features file etc.)
os.environ['HF_HUB_CACHE'] = f'{BIG}/hf/hub'
os.environ['HF_DATASETS_CACHE'] = f'{BIG}/hf/datasets'
os.environ['UV_CACHE_DIR'] = f'{BIG}/uv-cache'         # wheel cache — CUDA torch alone is ~2.5 GB
os.environ['UV_PYTHON_INSTALL_DIR'] = f'{BIG}/uv-pythons'
os.environ['PIP_CACHE_DIR'] = f'{BIG}/pip-cache'
# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline, which
# the training venv's matplotlib cannot load — torchmetrics imports
# matplotlib at startup and the whole training run dies on a ValueError.
# Agg (headless) is always available.
os.environ['MPLBACKEND'] = 'Agg'
print('caches ->', BIG)

In [ ]:
# Clone the repo ONTO THE BIG DISK. Every pipeline path in config.yaml
# (work_dir=./work etc.) resolves relative to config.yaml's own directory, so
# cloning under /kaggle/tmp relocates all ~30 GB of assets automatically —
# no config edits needed.
!rm -rf /kaggle/tmp/dead-space-bot
!git clone --depth 1 https://github.com/spacejunkindustries/dead-space-bot /kaggle/tmp/dead-space-bot
%cd /kaggle/tmp/dead-space-bot/training/wake
!ls

In [ ]:
# Python 3.10 venv via uv — Kaggle's host interpreter is 3.12 and
# piper-phonemize has no 3.12 wheels. --clear makes this cell re-runnable.
!pip install -q uv
!uv venv --clear --python 3.10 .venv
!uv pip install --python .venv/bin/python -r requirements-train.txt
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"
!df -h /kaggle/tmp

In [ ]:
# Stage 1 (the long one, 1-2h+): download assets (17.3 GB negative features,
# validation features, RIRs, AudioSet shard, FMA), synthesize ~30k
# 'hey cortana' positives + adversarial negatives, augment, compute features.
# Resumable: re-running skips completed stages.
# subprocess.run(check=True) so a failure STOPS the run — a bare ! line
# swallows exit codes and Kaggle would mark a dead run 'successful'.
import subprocess
subprocess.run(['.venv/bin/python', 'generate_samples.py', '--config', 'config.yaml'], check=True)

In [ ]:
# Reclaim space before training (safe: stage checks look at extracted dirs).
!uv cache clean
!rm -rf /kaggle/tmp/pip-cache
!df -h /kaggle/tmp

In [ ]:
# Stage 2: train the classifier head and print the threshold sweep.
# WRITE DOWN the flagged threshold — it becomes wake.threshold in aura.yaml.
# subprocess.run(check=True) so a failure STOPS the run — a bare ! line
# swallows exit codes and Kaggle would mark a dead run 'successful'.
import subprocess
subprocess.run(['.venv/bin/python', 'train.py', '--config', 'config.yaml'], check=True)

In [ ]:
# Stage 3: assemble the deployable 3-file ONNX chain.
import subprocess
subprocess.run(['.venv/bin/python', 'train.py', '--config', 'config.yaml',
                '--skip-train', '--skip-validate', '--bundle'], check=True)
!ls -lh work/deploy_bundle/

In [ ]:
# Surface the deliverables: copy ONLY the 3 small .onnx files into
# /kaggle/working so they persist as notebook Output (downloadable from the
# phone browser via the notebook's Output tab). Everything else is discarded
# with the session.
!mkdir -p /kaggle/working/deploy_bundle
!cp work/deploy_bundle/*.onnx /kaggle/working/deploy_bundle/
!ls -lh /kaggle/working/deploy_bundle
# Then from Termux:
#   scp melspectrogram.onnx embedding_model.onnx hey_cortana.onnx \
#       root@YOUR-DROPLET:/opt/aura/models/wake/
# and set in /etc/aura/aura.yaml:
#   wake.model: /opt/aura/models/wake/hey_cortana.onnx
#   wake.threshold: <the flagged sweep value>
# Hard check: the run is only a success if all three files made it.
from pathlib import Path
bundle = Path('/kaggle/working/deploy_bundle')
onnx = sorted(p.name for p in bundle.glob('*.onnx'))
assert len(onnx) == 3, f'training did NOT produce the 3-file bundle, got: {onnx}'
print('SUCCESS — download these from the Output tab:', onnx)